In [1]:
# Cell 1: Import necessary libraries and environment setup
import os
import torch
import torchaudio
import pandas as pd
import numpy as np
import gc
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import AutoProcessor, Wav2Vec2Model
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, accuracy_score

# Configure device and check GPU specifications
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("-" * 50)
print(f"🚀 Execution Device: {device}")

if torch.cuda.is_available():
    # Let's flex the hardware for your teammates!
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / (1024**3) # Convert to GB
    print(f"🔥 Hardware Model: {gpu_name}")
    print(f"📟 Dedicated VRAM: {gpu_memory:.2f} GB")
    print(f"✨ CUDA Capability: {torch.cuda.get_device_capability(0)}")
else:
    print("⚠️ Warning: Running on CPU. Performance might be limited.")
print("-" * 50)

# Global setting: Optimization for large-scale audio processing
# Note: We are using librosa for loading to bypass torchaudio/torchcodec version conflicts
import librosa 
print("✅ Environment initialized. Ready for FinQuest 2026.")

/home/jesse/Projects/MyEnvs/finquest_2026/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


--------------------------------------------------
🚀 Execution Device: cuda
🔥 Hardware Model: NVIDIA GeForce RTX 5080
📟 Dedicated VRAM: 15.46 GB
✨ CUDA Capability: (12, 0)
--------------------------------------------------
✅ Environment initialized. Ready for FinQuest 2026.


In [2]:
# Cell 2: Load Master Protocol and Perform Dataset Splitting (Train/Dev/Test)
# We partition the data using an 80/10/10 ratio for Training, Development, and Testing.
protocol_path = "./protocol/master_protocol.csv"
df = pd.read_csv(protocol_path)

# First split: Separate 80% for Training and 20% for the remaining pool (Temp)
# Note: 'stratify' ensures the Real/Fake ratio remains consistent across all sets.
train_df, temp_df = train_test_split(
    df, 
    test_size=0.2, 
    stratify=df['label'], 
    random_state=42
)

# Second split: Evenly divide the remaining 20% into Dev and Test sets (10% each of total)
dev_df, test_df = train_test_split(
    temp_df, 
    test_size=0.5, 
    stratify=temp_df['label'], 
    random_state=42
)

print("-" * 60)
print(f"📊 Dataset Statistics:")
print(f"Total Samples: {len(df)}")
print(f"Training Set : {len(train_df)} samples (80%)")
print(f"Validation Set: {len(dev_df)} samples (10%)")
print(f"Testing Set   : {len(test_df)} samples (10%)")
print("-" * 60)

# Verification of class balance for the team
print("✅ Stratification Check (Real vs Fake ratio):")
print(f"Train Label 1 Ratio: {train_df['label'].mean():.2%}")
print(f"Dev   Label 1 Ratio: {dev_df['label'].mean():.2%}")
print(f"Test  Label 1 Ratio: {test_df['label'].mean():.2%}")
print("-" * 60)

------------------------------------------------------------
📊 Dataset Statistics:
Total Samples: 143037
Training Set : 114429 samples (80%)
Validation Set: 14304 samples (10%)
Testing Set   : 14304 samples (10%)
------------------------------------------------------------
✅ Stratification Check (Real vs Fake ratio):
Train Label 1 Ratio: 50.25%
Dev   Label 1 Ratio: 50.25%
Test  Label 1 Ratio: 50.25%
------------------------------------------------------------


In [3]:
# Cell 3: Robust Audio Dataset Definition
import librosa
import torch
import torchaudio

class FinQuestDataset(Dataset):
    """
    Custom Dataset for Deepfake Detection.
    Uses librosa for high-stability audio loading and automatic resampling.
    """
    def __init__(self, dataframe, processor, max_length=64000):
        self.df = dataframe
        self.processor = processor
        self.max_length = max_length  # Default 64000 samples = 4 seconds @ 16kHz

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = row['path']
        label = row['label']

        try:
            # Step 1: Load audio using librosa for maximum compatibility
            # Forced resampling to 16,000Hz and mono conversion
            waveform, _ = librosa.load(path, sr=16000, mono=True)
            waveform = torch.from_numpy(waveform)
            
            # Step 2: Ensure uniform sequence length (Truncate or Zero-pad)
            if len(waveform) > self.max_length:
                waveform = waveform[:self.max_length]
            else:
                padding_needed = self.max_length - len(waveform)
                waveform = torch.nn.functional.pad(waveform, (0, padding_needed))

            # Step 3: Feature Extraction via Wav2Vec2 Processor
            inputs = self.processor(waveform, sampling_rate=16000, return_tensors="pt")
            
            return {
                "input_values": inputs.input_values.squeeze(0),
                "label": torch.tensor(label, dtype=torch.long)
            }
        except Exception as e:
            # Recursive Fallback: If a file is corrupted, skip to the next valid sample
            # This prevents the training loop from crashing during large-scale runs
            # print(f"Warning: Skipping corrupted file {path} - {e}")
            return self.__getitem__((idx + 1) % len(self.df))

print("✅ FinQuestDataset class defined with robust error handling and auto-resampling.")

✅ FinQuestDataset class defined with robust error handling and auto-resampling.


In [4]:
# Cell 4: Model Architecture Definition (Wav2Vec2 + Attentive Pooling + MLP)

class AttentivePooling(torch.nn.Module):
    """
    Self-Attentive Pooling layer to aggregate frame-level features into a 
    single utterance-level representation by focusing on relevant temporal segments.
    """
    def __init__(self, hidden_size):
        super().__init__()
        # Attention mechanism to compute importance scores for each frame
        self.attention = torch.nn.Sequential(
            torch.nn.Linear(hidden_size, 128),
            torch.nn.Tanh(),
            torch.nn.Linear(128, 1)
        )

    def forward(self, x):
        # Input shape x: (batch, seq_len, hidden_size)
        # Compute softmax-normalized attention weights across the time dimension
        attn_weights = torch.softmax(self.attention(x), dim=1)
        # Weighted summation of hidden states
        pooled = torch.sum(attn_weights * x, dim=1)
        return pooled

# Step 1: Load Pre-trained Wav2Vec 2.0 Base Engine
# This model provides robust acoustic feature extraction from raw audio.
processor = AutoProcessor.from_pretrained("facebook/wav2vec2-base")
backbone = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base").to(device)

# Step 2: Initialize Classification Head
# 768 is the hidden size for the wav2vec2-base transformer layers.
pooling = AttentivePooling(768).to(device)

# Multi-Layer Perceptron (MLP) for final binary classification (Real vs Fake)
classifier = torch.nn.Sequential(
    torch.nn.Linear(768, 256),
    torch.nn.ReLU(),
    torch.nn.Dropout(0.3),  # Regularization to prevent overfitting
    torch.nn.Linear(256, 2)
).to(device)

print(f"✅ Model architecture initialized on {device}.")
print(f"🔹 Backbone: Wav2Vec2-Base (Frozen/Fine-tuned)")
print(f"🔹 Aggregation: Self-Attentive Pooling")
print(f"🔹 Output: Binary Logits [Fake, Real]")

Loading weights: 100%|██████████| 211/211 [00:00<00:00, 87269.32it/s]
Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model architecture initialized on cuda.
🔹 Backbone: Wav2Vec2-Base (Frozen/Fine-tuned)
🔹 Aggregation: Self-Attentive Pooling
🔹 Output: Binary Logits [Fake, Real]


In [5]:
# Cell 5: Training Hyperparameters & Optimizer Configuration (LLRD Version)

# Initialize Training and Validation Datasets
train_dataset = FinQuestDataset(train_df, processor)
dev_dataset   = FinQuestDataset(dev_df, processor)

# DataLoaders: batch_size=4 is used for RTX 5080 VRAM efficiency. 
# Increasing batch_size to 16 may provide more stable gradient estimates if VRAM allows.
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=4)
dev_loader   = DataLoader(dev_dataset, batch_size=16, num_workers=4)

# Layer-wise Learning Rate Decay (LLRD) Strategy:
# We apply differential learning rates to preserve pre-trained knowledge while 
# allowing the classification head to adapt quickly to deepfake-specific patterns.
optimizer = torch.optim.AdamW([
    # 1. Low-level CNN Feature Extractor: 2e-6 (Keep low to protect basic acoustic filters)
    {"params": backbone.feature_extractor.parameters(), "lr": 2e-6}, 
    
    # 2. Mid-level Transformer Encoder: 8e-6 (Fine-tune contextual representations)
    {"params": backbone.encoder.parameters(), "lr": 8e-6},
    
    # 3. High-level Pooling & Classifier Heads: 8e-5 (Focus on learning new spoofing artifacts)
    {"params": pooling.parameters(), "lr": 8e-5},
    {"params": classifier.parameters(), "lr": 8e-5},
], weight_decay=0.01)

# Standard Loss Function for Binary Classification (Real vs Fake)
criterion = torch.nn.CrossEntropyLoss()

print("-" * 60)
print("🚀 Training Configuration Summary:")
print(f"🔹 Optimizer    : AdamW with LLRD")
print(f"🔹 Batch Size   : {train_loader.batch_size}")
print(f"🔹 CNN LR       : 2e-6")
print(f"🔹 Encoder LR   : 8e-6")
print(f"🔹 Head LR      : 8e-5")
print(f"🔹 Weight Decay : 0.01")
print("-" * 60)
print("✅ Optimization engine ready for fine-tuning.")

------------------------------------------------------------
🚀 Training Configuration Summary:
🔹 Optimizer    : AdamW with LLRD
🔹 Batch Size   : 16
🔹 CNN LR       : 2e-6
🔹 Encoder LR   : 8e-6
🔹 Head LR      : 8e-5
🔹 Weight Decay : 0.01
------------------------------------------------------------
✅ Optimization engine ready for fine-tuning.


In [6]:
# Cell 6: Training & Validation Loop (Warmup & LLRD Optimized Edition)
import os
import warnings
import logging
import torch
import numpy as np
import csv
from tqdm import tqdm
from sklearn.metrics import roc_curve, accuracy_score
# --- 新增：引入 Transformers 的 Scheduler ---
from transformers import get_cosine_schedule_with_warmup

# ==============================================================================
# 🚀 ULTIMATE NOISE SUPPRESSION
# ==============================================================================
os.environ["PYTHONWARNINGS"] = "ignore"
warnings.filterwarnings("ignore") 
warnings.filterwarnings("ignore", category=UserWarning, module="librosa")
warnings.filterwarnings("ignore", category=FutureWarning)
logging.getLogger('librosa').setLevel(logging.ERROR)
logging.getLogger('audioread').setLevel(logging.ERROR)

# Ensure the model and log storage directory exists
MODEL_DIR = "./model"
LOG_FILE = os.path.join(MODEL_DIR, "training_log.csv")
os.makedirs(MODEL_DIR, exist_ok=True)

# --- Initialize CSV Log File ---
if not os.path.exists(LOG_FILE):
    with open(LOG_FILE, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(["epoch", "train_loss", "dev_acc", "dev_eer", "is_best"])

def save_metrics_to_csv(epoch, loss, acc, eer, is_best):
    with open(LOG_FILE, 'a', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([epoch, f"{loss:.6f}", f"{acc:.4f}", f"{eer:.4f}", "YES" if is_best else "NO"])

def evaluate(model_base, pool_layer, clf_layer, loader):
    model_base.eval()
    pool_layer.eval()
    clf_layer.eval()
    all_scores, all_labels = [], []
    
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        with torch.no_grad():
            for batch in tqdm(loader, desc="Validating", leave=False):
                inputs = batch["input_values"].to(device)
                labels = batch["label"].to(device)
                
                outputs = model_base(inputs).last_hidden_state
                pooled = pool_layer(outputs)
                logits = clf_layer(pooled)
                
                probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
                all_scores.extend(probs)
                all_labels.extend(labels.cpu().numpy())
            
    all_labels = np.array(all_labels)
    all_scores = np.array(all_scores)
    
    fpr, tpr, _ = roc_curve(all_labels, all_scores, pos_label=1)
    fnr = 1 - tpr
    eer = fpr[np.nanargmin(np.abs(fnr - fpr))]
    acc = accuracy_score(all_labels, all_scores > 0.5)
    return acc, eer

# --- 🚀 新增：Warmup Scheduler 配置 ---
max_epochs = 20    
patience = 3       
num_training_steps = len(train_loader) * max_epochs
# 预热步数设为总步数的 10%，让模型在 BS=16 的大步长下平稳起步
num_warmup_steps = int(0.1 * num_training_steps)

scheduler = get_cosine_schedule_with_warmup(
    optimizer, 
    num_warmup_steps=num_warmup_steps, 
    num_training_steps=num_training_steps
)

# --- Training Configuration & Early Stopping ---
best_eer = 1.0     
counter = 0        

print(f"🏁 FinQuest 2026: Training initiated. BS={train_loader.batch_size}, Warmup={num_warmup_steps} steps.")
print(f"📈 Metrics will be saved to: {LOG_FILE}")

for epoch in range(max_epochs):
    backbone.train()
    pooling.train()
    classifier.train()
    total_loss = 0
    
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        pbar = tqdm(train_loader, desc=f"Epoch {epoch}")
        for batch in pbar:
            optimizer.zero_grad()
            
            inputs = batch["input_values"].to(device)
            labels = batch["label"].to(device)
            
            outputs = backbone(inputs).last_hidden_state
            pooled = pooling(outputs)
            logits = classifier(pooled)
            
            loss = criterion(logits, labels)
            loss.backward()
            
            # --- 🚀 关键修改：更新参数的同时更新学习率 ---
            optimizer.step()
            scheduler.step() 
            
            total_loss += loss.item()
            # 在 tqdm 中显示当前最后一位参数组（即 Classifier）的学习率，方便监控
            current_lr = optimizer.param_groups[-1]['lr']
            pbar.set_postfix(loss=f"{loss.item():.4f}", lr=f"{current_lr:.2e}")

    # --- Validation Phase ---
    dev_acc, dev_eer = evaluate(backbone, pooling, classifier, dev_loader)
    avg_loss = total_loss / len(train_loader)
    
    is_best = dev_eer < best_eer
    save_metrics_to_csv(epoch, avg_loss, dev_acc, dev_eer, is_best)
    
    print(f"\n📊 [Epoch {epoch} Summary]")
    print(f"   - Training Loss : {avg_loss:.6f}")
    print(f"   - Dev Accuracy  : {dev_acc:.4f}")
    print(f"   - Dev EER       : {dev_eer:.4f}")
    
    if is_best:
        best_eer = dev_eer
        counter = 0  
        save_path = os.path.join(MODEL_DIR, "best_deepfake_model.pth")
        
        clean_state_dict = {
            'backbone': backbone.state_dict(),
            'pooling': pooling.state_dict(),
            'classifier': classifier.state_dict()
        }
        
        torch.save(clean_state_dict, save_path)
        print(f"🥇 New Best Model! Performance improved. Pure weights saved to {save_path}")
    else:
        counter += 1
        print(f"⏱ No improvement. Counter: {counter}/{patience}")
    
    if counter >= patience:
        print(f"🛑 Early Stopping triggered. Best EER: {best_eer:.4f}")
        break
    print("-" * 50)

print("\n✨ Training session complete. All logs are in CSV format.")

🏁 FinQuest 2026: Training initiated. BS=16, Warmup=14304 steps.
📈 Metrics will be saved to: ./model/training_log.csv


Epoch 0: 100%|██████████| 7152/7152 [19:57<00:00,  5.97it/s, loss=0.0074, lr=4.00e-05]



📊 [Epoch 0 Summary]
   - Training Loss : 0.195933
   - Dev Accuracy  : 0.9899
   - Dev EER       : 0.0086
🥇 New Best Model! Performance improved. Pure weights saved to ./model/best_deepfake_model.pth
--------------------------------------------------


Epoch 1: 100%|██████████| 7152/7152 [19:53<00:00,  5.99it/s, loss=0.0053, lr=8.00e-05]



📊 [Epoch 1 Summary]
   - Training Loss : 0.022153
   - Dev Accuracy  : 0.9871
   - Dev EER       : 0.0069
🥇 New Best Model! Performance improved. Pure weights saved to ./model/best_deepfake_model.pth
--------------------------------------------------


Epoch 2: 100%|██████████| 7152/7152 [20:17<00:00,  5.87it/s, loss=0.0021, lr=7.94e-05]


... (9 outputs left out)

Epoch 7: 100%|██████████| 7152/7152 [20:12<00:00,  5.90it/s, loss=0.0000, lr=6.00e-05]



📊 [Epoch 7 Summary]
   - Training Loss : 0.002469
   - Dev Accuracy  : 0.9981
   - Dev EER       : 0.0010
⏱ No improvement. Counter: 2/3
--------------------------------------------------


Epoch 8: 100%|██████████| 7152/7152 [20:00<00:00,  5.96it/s, loss=0.0000, lr=5.37e-05]
                                                             


📊 [Epoch 8 Summary]
   - Training Loss : 0.001795
   - Dev Accuracy  : 0.9986
   - Dev EER       : 0.0013
⏱ No improvement. Counter: 3/3
🛑 Early Stopping triggered. Best EER: 0.0010

✨ Training session complete. All logs are in CSV format.


In [7]:
# Cell 7: Full Test Evaluation and Model Source Attribution Report
import librosa
from sklearn.metrics import roc_auc_score, confusion_matrix

# 1. Load the optimal weights from the model directory
print("🔄 Loading best model weights for final evaluation...")
checkpoint = torch.load("model/best_deepfake_model.pth", weights_only=False)
backbone.load_state_dict(checkpoint['backbone'])
pooling.load_state_dict(checkpoint['pooling'])
classifier.load_state_dict(checkpoint['classifier'])

backbone.eval()
pooling.eval()
classifier.eval()

all_scores = []
all_labels = []
source_results = {} # Metrics storage for cross-algorithm traceability

print(f"🔍 Running full inference on {len(test_df)} unseen test samples...")

with torch.no_grad():
    for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Inference Phase"):
        try:
            # Step 1: Pre-processing
            wav, _ = librosa.load(row['path'], sr=16000, mono=True)
            inputs = processor(wav, sampling_rate=16000, return_tensors="pt").input_values.to(device)
            
            # Step 2: Forward pass through the inference pipeline
            out = backbone(inputs).last_hidden_state
            pooled = pooling(out)
            logits = classifier(pooled)
            
            # Compute probability for Class 1 (Real)
            prob = torch.softmax(logits, dim=1)[0, 1].item()
            
            label = row['label']
            source = row['model_name'] # Source algorithm (e.g., MelGAN, WaveGAN)
            
            all_scores.append(prob)
            all_labels.append(label)
            
            # Step 3: Attribution tracking
            if source not in source_results:
                source_results[source] = {"total": 0, "correct": 0}
            
            pred = 1 if prob > 0.5 else 0
            source_results[source]["total"] += 1
            if pred == label:
                source_results[source]["correct"] += 1
                
        except Exception as e:
            # Skip corrupted samples if any
            continue

# --- Final Metric Computation ---
all_labels = np.array(all_labels)
all_scores = np.array(all_scores)

# Accuracy Calculation
final_acc = accuracy_score(all_labels, all_scores > 0.5)

# Equal Error Rate (EER) Calculation
fpr, tpr, thresholds = roc_curve(all_labels, all_scores, pos_label=1)
fnr = 1 - tpr
eer = fpr[np.nanargmin(np.abs(fnr - fpr))]

# Area Under the ROC Curve (AUC)
auc_score = roc_auc_score(all_labels, all_scores)

# --- Final Performance Report ---
print("\n" + "="*60)
print("🚀 FINQUEST 2026: COMPREHENSIVE TEST ANALYSIS REPORT")
print("="*60)
print(f"✨ Overall Accuracy       : {final_acc:.2%}")
print(f"✨ Equal Error Rate (EER) : {eer:.4f}")
print(f"✨ Area Under Curve (AUC) : {auc_score:.4f}")
print("-" * 60)
print(f"{'Synthetic Source':<18} | {'Count':<8} | {'Accuracy':<10}")
print("-" * 60)

for src, res in source_results.items():
    src_acc = res['correct'] / res['total']
    # If the source is 'original', it refers to Real samples
    source_label = "Real Samples" if src == "original" else f"Fake ({src})"
    print(f"{source_label:<18} | {res['total']:<8} | {src_acc:.2%}")

print("-" * 60)

# --- Traceability Sample Display ---
print("\n📋 Random Sample Traceability Check:")
print(f"{'Filename':<18} | {'Pred':<6} | {'Conf.':<8} | {'Actual':<6} | {'Algorithm'}")
print("-" * 60)

# Select a subset for visualization
test_samples = test_df.sample(min(10, len(test_df)))
for _, row in test_samples.iterrows():
    wav, _ = librosa.load(row['path'], sr=16000, mono=True)
    inputs = processor(wav, sampling_rate=16000, return_tensors="pt").input_values.to(device)
    with torch.no_grad():
        out = backbone(inputs).last_hidden_state
        logits = classifier(pooling(out))
        prob = torch.softmax(logits, dim=1)[0, 1].item()
    
    pred_str = "REAL" if prob > 0.5 else "FAKE"
    actual_str = "REAL" if row['label'] == 1 else "FAKE"
    fname = os.path.basename(row['path'])[:15] + ".."
    
    print(f"{fname:<18} | {pred_str:<6} | {prob:>7.2%} | {actual_str:<6} | {row['model_name']}")

# Clear memory for subsequent tasks
torch.cuda.empty_cache()
print("\n✅ Final evaluation complete.")

🔄 Loading best model weights for final evaluation...
🔍 Running full inference on 14304 unseen test samples...


Inference Phase: 100%|██████████| 14304/14304 [01:55<00:00, 124.20it/s]



🚀 FINQUEST 2026: COMPREHENSIVE TEST ANALYSIS REPORT
✨ Overall Accuracy       : 99.83%
✨ Equal Error Rate (EER) : 0.0017
✨ Area Under Curve (AUC) : 1.0000
------------------------------------------------------------
Synthetic Source   | Count    | Accuracy  
------------------------------------------------------------
Fake (fastspeech2_pwg) | 1645     | 100.00%
Real Samples       | 1998     | 99.95%
Fake (for-original) | 1395     | 100.00%
Fake (for-2seconds) | 1004     | 99.80%
Fake (unknown_wild) | 3172     | 99.78%
Fake (for-norm)    | 2232     | 99.46%
Fake (for-rerecorded) | 767      | 100.00%
Fake (wavegan)     | 449      | 99.78%
Fake (A03)         | 38       | 100.00%
Fake (large)       | 185      | 100.00%
Fake (melgan)      | 828      | 100.00%
Fake (A01)         | 44       | 100.00%
Fake (hifiGAN)     | 188      | 99.47%
Fake (A04)         | 39       | 97.44%
Fake (A02)         | 41       | 100.00%
Fake (waveglow)    | 194      | 100.00%
Fake (A06)         | 37       | 100.0

In [11]:
# Cell 8: Multi-Scenario Inference & Evaluation
import os
import torch
import torch.nn as nn
import librosa
import numpy as np
import pandas as pd
from tqdm import tqdm
from pathlib import Path
from transformers import Wav2Vec2Model, Wav2Vec2FeatureExtractor

# ==============================================================================
# 1. Environment & Path Configuration
# ==============================================================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TARGET_SR = 16000
BASE_PATH = Path("/home/jesse/Projects/MyProjects/2026_FinQuest/Finquest_Wav2Vec_Deepfake/Fake_or_Real")
MODEL_PATH = "/home/jesse/Projects/MyProjects/2026_FinQuest/Finquest_Wav2Vec_Deepfake/model/best_deepfake_model.pth"
MODEL_NAME = "facebook/wav2vec2-base" 

# ==============================================================================
# 2. Model Architecture (Matching Training Cell 7)
# ==============================================================================
class FinQuestDeepfakeModel(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.wav2vec2 = Wav2Vec2Model.from_pretrained(model_name)
        # Sequential structure: 0 (Linear), 1 (ReLU), 2 (Dropout), 3 (Linear)
        self.classifier = nn.Sequential(
            nn.Linear(768, 256), 
            nn.ReLU(),           
            nn.Dropout(0.1),     
            nn.Linear(256, 2)    
        )

    def forward(self, input_values):
        outputs = self.wav2vec2(input_values)
        # Using the CLS-like representation (first token) for classification
        hidden_states = outputs.last_hidden_state[:, 0, :] 
        return self.classifier(hidden_states)

# ==============================================================================
# 3. Loading Logic (Standardized Dictionary Support)
# ==============================================================================
print(f"📦 Loading model architecture and weights on {DEVICE}...")

processor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_NAME)
model = FinQuestDeepfakeModel(MODEL_NAME)

# Load pure state_dict
checkpoint = torch.load(MODEL_PATH, map_location=DEVICE, weights_only=False)

# Mapping logic for combined weights
if isinstance(checkpoint, dict) and "backbone" in checkpoint:
    print("🔍 Mapping weights from structured dictionary...")
    model.wav2vec2.load_state_dict(checkpoint["backbone"])
    if "classifier" in checkpoint:
        # strict=False allows loading even if dropout indices slightly differ
        model.classifier.load_state_dict(checkpoint["classifier"], strict=False)
else:
    # Standard load for direct state_dicts
    model.load_state_dict(checkpoint, strict=False)

model.to(DEVICE)
model.eval()
print("✅ Weights successfully aligned!")

# ==============================================================================
# 4. Inference Function
# ==============================================================================
def predict_audio(audio_path, model, processor, device):
    try:
        audio, _ = librosa.load(audio_path, sr=TARGET_SR)
        inputs = processor(audio, return_tensors="pt", sampling_rate=TARGET_SR)
        input_values = inputs.input_values.to(device)
        
        with torch.no_grad():
            logits = model(input_values)
            probs = torch.softmax(logits, dim=-1)
            # Assumption: Index 1 is REAL. 
            # If accuracy is very low (e.g. ~15%), try changing to probs[0][0]
            return probs[0][1].item() 
    except Exception as e:
        return 0.5

# ==============================================================================
# 5. Multi-Scenario Execution
# ==============================================================================
SCENARIOS = ["for-2seconds", "for-norm", "for-original", "for-rerecorded"]
summary_report = {}

print("-" * 60)
for scenario in SCENARIOS:
    test_path = BASE_PATH / scenario / "testing"
    if not test_path.exists():
        continue
        
    print(f"▶️ Analyzing Scenario: [{scenario}]")
    correct_count = 0
    total_count = 0
    
    for label in ["real", "fake"]:
        folder = test_path / label
        if not folder.exists(): continue
        
        audio_files = list(folder.glob("*.wav"))
        for f in tqdm(audio_files, desc=f"  Evaluating {label}", leave=False):
            score = predict_audio(f, model, processor, DEVICE)
            
            is_pred_real = score > 0.5
            is_actually_real = (label == "real")
            
            if is_pred_real == is_actually_real:
                correct_count += 1
            total_count += 1
            
    if total_count > 0:
        accuracy = (correct_count / total_count) * 100
        summary_report[scenario] = f"{accuracy:.2f}%"
        print(f"✨ Accuracy for [{scenario}]: {accuracy:.2f}%")

# ==============================================================================
# 6. Final Report Summary
# ==============================================================================
print("\n" + "="*60)
print("📊 FINQUEST 2026: FINAL SCENARIO REPORT")
print("-" * 60)
for scene, acc in summary_report.items():
    print(f"Scenario: {scene:<15} | Accuracy: {acc}")
print("="*60)

📦 Loading model architecture and weights on cuda...


Loading weights: 100%|██████████| 211/211 [00:00<00:00, 59447.72it/s]
Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🔍 Mapping weights from structured dictionary...
✅ Weights successfully aligned!
------------------------------------------------------------
▶️ Analyzing Scenario: [for-2seconds]


✨ Accuracy for [for-2seconds]: 68.75%
▶️ Analyzing Scenario: [for-norm]


✨ Accuracy for [for-norm]: 66.57%
▶️ Analyzing Scenario: [for-original]


✨ Accuracy for [for-original]: 78.87%
▶️ Analyzing Scenario: [for-rerecorded]


✨ Accuracy for [for-rerecorded]: 56.00%

📊 FINQUEST 2026: FINAL SCENARIO REPORT
------------------------------------------------------------
Scenario: for-2seconds    | Accuracy: 68.75%
Scenario: for-norm        | Accuracy: 66.57%
Scenario: for-original    | Accuracy: 78.87%
Scenario: for-rerecorded  | Accuracy: 56.00%


In [14]:
# Cell 8: Multi-Scenario Inference & Evaluation (PERFECT ALIGNMENT VERSION)
import os
import torch
import torch.nn as nn
import librosa
import numpy as np
import pandas as pd
from tqdm import tqdm
from pathlib import Path
from transformers import Wav2Vec2Model, Wav2Vec2FeatureExtractor

# ==============================================================================
# 1. Environment & Configuration (MATCHING CELL 3)
# ==============================================================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TARGET_SR = 16000
MAX_LENGTH = 64000  # 🚀 CRITICAL: Must match Cell 3's max_length

BASE_PATH = Path("/home/jesse/Projects/MyProjects/2026_FinQuest/Finquest_Wav2Vec_Deepfake/Fake_or_Real")
MODEL_PATH = "/home/jesse/Projects/MyProjects/2026_FinQuest/Finquest_Wav2Vec_Deepfake/model/best_deepfake_model.pth"
MODEL_NAME = "facebook/wav2vec2-base"

# ==============================================================================
# 2. Model Architecture (EXACT MATCH)
# ==============================================================================
class AttentivePooling(torch.nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.attention = torch.nn.Sequential(
            torch.nn.Linear(hidden_size, 128),
            torch.nn.Tanh(),
            torch.nn.Linear(128, 1)
        )
    def forward(self, x):
        attn_weights = torch.softmax(self.attention(x), dim=1)
        pooled = torch.sum(attn_weights * x, dim=1)
        return pooled

class FinQuestDeepfakeModel(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.wav2vec2 = Wav2Vec2Model.from_pretrained(model_name)
        self.pooling = AttentivePooling(768)
        self.classifier = nn.Sequential(
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 2)
        )
    def forward(self, input_values):
        outputs = self.wav2vec2(input_values).last_hidden_state
        pooled = self.pooling(outputs) 
        return self.classifier(pooled)

# ==============================================================================
# 3. Loading Weights
# ==============================================================================
processor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_NAME)
model = FinQuestDeepfakeModel(MODEL_NAME)
checkpoint = torch.load(MODEL_PATH, map_location=DEVICE)

model.wav2vec2.load_state_dict(checkpoint["backbone"])
model.pooling.load_state_dict(checkpoint["pooling"])
model.classifier.load_state_dict(checkpoint["classifier"])

model.to(DEVICE)
model.eval()
print("✅ Weights and Architecture aligned. Inference mode active.")

# ==============================================================================
# 4. Corrected Inference Function (REPLICATING CELL 3 LOGIC)
# ==============================================================================
def predict_audio(audio_path, model, processor, device):
    try:
        # Step 1: Load (Matches Cell 3)
        waveform, _ = librosa.load(audio_path, sr=TARGET_SR, mono=True)
        waveform = torch.from_numpy(waveform)
        
        # Step 2: 🚀 THE MISSING PIECE - Manual Padding/Truncation
        # Must strictly follow the training dataset behavior
        if len(waveform) > MAX_LENGTH:
            waveform = waveform[:MAX_LENGTH]
        else:
            padding_needed = MAX_LENGTH - len(waveform)
            waveform = torch.nn.functional.pad(waveform, (0, padding_needed))
        
        # Step 3: Feature Extraction
        inputs = processor(waveform, sampling_rate=TARGET_SR, return_tensors="pt")
        input_values = inputs.input_values.to(device)
        
        with torch.no_grad():
            logits = model(input_values)
            probs = torch.softmax(logits, dim=-1)
            return probs[0][1].item() # Real Prob
    except Exception as e:
        return 0.5

# ==============================================================================
# 5. Multi-Scenario Execution
# ==============================================================================
SCENARIOS = ["for-2seconds", "for-norm", "for-original", "for-rerecorded"]
summary_report = {}

for scenario in SCENARIOS:
    test_path = BASE_PATH / scenario / "testing"
    if not test_path.exists(): continue
        
    print(f"\n▶️ Analyzing Scenario: [{scenario}]")
    correct_count, total_count = 0, 0
    
    for label in ["real", "fake"]:
        folder = test_path / label
        if not folder.exists(): continue
        
        for f in tqdm(list(folder.glob("*.wav")), desc=f"  {label}", leave=False):
            score = predict_audio(f, model, processor, DEVICE)
            if (score > 0.5) == (label == "real"):
                correct_count += 1
            total_count += 1

            # 在 score = predict_audio(...) 后面加
            if score > 0.4 and score < 0.6:
                print(f"⚠️ Marginal decision for {f.name}: {score:.4f}")
                    
    if total_count > 0:
        acc = (correct_count / total_count) * 100
        summary_report[scenario] = f"{acc:.2f}%"
        print(f"✨ Accuracy for [{scenario}]: {acc:.2f}%")

print("\n" + "="*60 + "\n📊 FINAL REPORT\n" + "-"*60)
for scene, acc in summary_report.items():
    print(f"Scenario: {scene:<15} | Accuracy: {acc}")
print("="*60)

Loading weights: 100%|██████████| 211/211 [00:00<00:00, 63340.83it/s]
Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Weights and Architecture aligned. Inference mode active.

▶️ Analyzing Scenario: [for-2seconds]


  fake:  26%|██▌       | 140/544 [00:00<00:02, 149.56it/s]

⚠️ Marginal decision for file1339.wav_16k.wav_norm.wav_mono.wav_silence.wav_2sec.wav: 0.4583
⚠️ Marginal decision for file899.wav_16k.wav_norm.wav_mono.wav_silence.wav_2sec.wav: 0.4020


  fake:  62%|██████▎   | 340/544 [00:02<00:01, 149.79it/s]

⚠️ Marginal decision for file2283.wav_16k.wav_norm.wav_mono.wav_silence.wav_2sec.wav: 0.5123


... (140 outputs left out)

⚠️ Marginal decision for recording13325.wav_norm_mono.wav: 0.5544
⚠️ Marginal decision for recording13488.wav_norm_mono.wav: 0.5881


  fake:  97%|█████████▋| 396/408 [00:02<00:00, 149.10it/s]

⚠️ Marginal decision for recording13416.wav_norm_mono.wav: 0.4136
⚠️ Marginal decision for recording13061.wav_norm_mono.wav: 0.5779


✨ Accuracy for [for-rerecorded]: 67.03%

📊 FINAL REPORT
------------------------------------------------------------
Scenario: for-2seconds    | Accuracy: 77.57%
Scenario: for-norm        | Accuracy: 79.11%
Scenario: for-original    | Accuracy: 81.20%
Scenario: for-rerecorded  | Accuracy: 67.03%
